## Data Configuration

In [16]:
# ============================================
# DATA CONFIGURATION - Change mode here
# ============================================
MODE = 'TRAIN'  # Options: 'TRAIN' or 'TEST'

TRAIN_FILES = [
    '../data/raw/2021_Jan-Mar.csv',
    '../data/raw/2021_Aprl-Jun.csv',
    '../data/raw/2021_Jul-Sep.csv',
]

TEST_FILES = [
    '../data/raw/2021_Oct-Dec.csv'
]

print(f"🔧 MODE: {MODE}")

🔧 MODE: TRAIN


In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [18]:
column_names = [
    'Procuring Entity (PE)', 'Region', 'Province', 'City/Municipality', 
    'Government Branch', 'PE Organization Type', 'PE Organization Type (Grouped)', 
    'Bid Reference No.', 'Notice Title', 'Classification', 'Procurement Mode', 
    'Business Category', 'Funding Source', 'Funding Instrument', 'Trade Agreement', 
    'Approved Budget of the Contract', 'Published Date', 'Closing Date', 
    'Area of Delivery', 'Contract Duration', 'Calendar Type', 'Line Item No', 
    'Item Name', 'Item Description', 'Quantity', 'UOM', 'Item Budget', 
    'Bid Notice Status', 'Award Reference No.', 'Award Title', 'UNSPSC Code', 
    'UNSPSC Description', 'Published Date(Award)', 'Award Date', 'Contract Amount', 
    'Award Notice Status', 'Notice to Proceed Date', 'Contract Effectivity Date', 
    'Contract End Date', 'Awardee Organization Name', 'Country of Awardee', 
    'Region of Awardee', 'Province of Awardee', 'City/Municipality of Awardee', 
    'Awardee Size', 'Awardee Joint Venture'
]

files = TRAIN_FILES if MODE == 'TRAIN' else TEST_FILES
dfs = []

for file in files:
    df_temp = pd.read_csv(file, names=column_names)
    dfs.append(df_temp)
    print(f"  ✓ {Path(file).name}: {len(df_temp):,} rows")


df = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Total: {len(df):,} rows")

/tmp/ipykernel_28145/1189342226.py:21: DtypeWarning: Columns (0: Bid Reference No., 1: Approved Budget of the Contract, 2: Contract Duration, 3: Line Item No, 4: Item Name, 5: Item Description, 6: Quantity, 7: UOM, 8: Item Budget, 9: Award Reference No., 10: Award Title, 11: UNSPSC Code, 12: UNSPSC Description, 13: Published Date(Award), 14: Award Date, 15: Contract Amount, 16: Award Notice Status, 17: Notice to Proceed Date, 18: Contract Effectivity Date, 19: Contract End Date, 20: Awardee Organization Name, 21: Country of Awardee, 22: Region of Awardee, 23: Province of Awardee, 24: City/Municipality of Awardee, 25: Awardee Size, 26: Awardee Joint Venture) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(file, names=column_names)


  ✓ 2021_Jan-Mar.csv: 456,115 rows
  ✓ 2021_Aprl-Jun.csv: 468,623 rows
  ✓ 2021_Jul-Sep.csv: 478,312 rows

📊 Total: 1,403,050 rows


## 2. Data Quality Filter

Remove corrupted rows with invalid financial/date values.

In [19]:
original_count = len(df)
print(f"Raw: {original_count:,} rows")

# Convert dates and financial columns
df['Published Date'] = pd.to_datetime(df['Published Date'], errors='coerce')
df['Closing Date'] = pd.to_datetime(df['Closing Date'], errors='coerce')
df['Award Date'] = pd.to_datetime(df['Award Date'], errors='coerce')
df['Approved Budget of the Contract'] = pd.to_numeric(df['Approved Budget of the Contract'], errors='coerce')
df['Contract Amount'] = pd.to_numeric(df['Contract Amount'], errors='coerce')

# Quality checks (vectorized)
valid_finance = (
    df['Approved Budget of the Contract'].notna() & 
    df['Contract Amount'].notna() &
    (df['Approved Budget of the Contract'] >= 0) &
    (df['Contract Amount'] >= 0)
)



# Combine filters
clean_mask = valid_finance

# Apply filter ONCE and store in df
df = df[clean_mask].copy()

print(f"Clean: {len(df):,} rows")
print(f"Dropped: {original_count - len(df):,} ({(original_count - len(df))/original_count*100:.1f}%)")

# Optional: Show breakdown of what was dropped
print(f"\nBreakdown of dropped rows:")
print(f"  Invalid finance: {(~valid_finance).sum():,}")

Raw: 1,403,050 rows


/tmp/ipykernel_28145/1158045103.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Published Date'] = pd.to_datetime(df['Published Date'], errors='coerce')
/tmp/ipykernel_28145/1158045103.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Closing Date'] = pd.to_datetime(df['Closing Date'], errors='coerce')
/tmp/ipykernel_28145/1158045103.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Award Date'] = pd.to_datetime(df['Award Date'], errors='coerce')


Clean: 746,849 rows
Dropped: 656,201 (46.8%)

Breakdown of dropped rows:
  Invalid finance: 656,201


## 3. Deduplicate to Contract Level

Same bid appears multiple times (different UNSPSC codes). Keep first occurrence.

In [20]:
## 3. Smart Aggregation with Feature Engineering
print("=== CONTRACT AGGREGATION WITH FEATURES ===")

# First, get contract-level aggregates
contract_agg = df.groupby('Bid Reference No.').agg({
    # Financial sums
    'Approved Budget of the Contract': 'sum',
    'Contract Amount': 'sum',
    'Item Budget': 'sum',
    
    # First occurrences (stable fields)
    'Published Date': 'first',
    'Closing Date': 'first',
    'Award Date': 'first',
    'Procuring Entity (PE)': 'first',
    'Procurement Mode': 'first',
    'Funding Instrument': 'first',
    'Awardee Organization Name': 'first',
    'Region': 'first',
    'Province': 'first',
    'Awardee Size': 'first',
    
    'Notice Title': 'first', 
    'UNSPSC Description': 'first',  
    'Contract Duration': 'first',


    # Count of items
    'Line Item No': 'count',
    
    # Diversity metrics
    'UNSPSC Code': lambda x: len(set(x)),  # Number of unique UNSPSC codes
})

# Add new columns but KEEP original names
contract_agg['item_count'] = contract_agg['Line Item No']  # Copy the count
contract_agg['unique_unspsc_count'] = contract_agg['UNSPSC Code']  # Copy the unique count

# Drop the aggregation columns that might cause confusion
contract_agg = contract_agg.drop(columns=['Line Item No'])

print(f"Contracts: {len(contract_agg):,}")
print(f"Items per contract - mean: {contract_agg['item_count'].mean():.1f}, max: {contract_agg['item_count'].max()}")
print(f"Multi-item contracts: {(contract_agg['item_count'] > 1).sum():,} ({(contract_agg['item_count'] > 1).mean()*100:.1f}%)")

=== CONTRACT AGGREGATION WITH FEATURES ===
Contracts: 127,479
Items per contract - mean: 5.9, max: 35907
Multi-item contracts: 127,477 (100.0%)


In [21]:
# Reset index to make 'Bid Reference No.' a column again
contract_agg = contract_agg.reset_index()
print(f"Columns after reset: {contract_agg.columns.tolist()}")

df_dedup = contract_agg.copy()  # Now df_dedup has the smart aggregation!

print(f"\n✅ Smart aggregation complete: {len(df_dedup):,} unique contracts")
print(f"   Removed: {len(df) - len(df_dedup):,} duplicate line items")

# Show all columns to verify
print("\nColumns preserved:")
print(df_dedup.columns.tolist())

Columns after reset: ['Bid Reference No.', 'Approved Budget of the Contract', 'Contract Amount', 'Item Budget', 'Published Date', 'Closing Date', 'Award Date', 'Procuring Entity (PE)', 'Procurement Mode', 'Funding Instrument', 'Awardee Organization Name', 'Region', 'Province', 'Awardee Size', 'Notice Title', 'UNSPSC Description', 'Contract Duration', 'UNSPSC Code', 'item_count', 'unique_unspsc_count']

✅ Smart aggregation complete: 127,479 unique contracts
   Removed: 619,370 duplicate line items

Columns preserved:
['Bid Reference No.', 'Approved Budget of the Contract', 'Contract Amount', 'Item Budget', 'Published Date', 'Closing Date', 'Award Date', 'Procuring Entity (PE)', 'Procurement Mode', 'Funding Instrument', 'Awardee Organization Name', 'Region', 'Province', 'Awardee Size', 'Notice Title', 'UNSPSC Description', 'Contract Duration', 'UNSPSC Code', 'item_count', 'unique_unspsc_count']


## 4. Create Derived Features

**Audit Risk Indicators:**
- Price variance %
- Bidding duration (days)
- Award speed (days)

In [22]:
# Price variance % = (Contract - ABC) / ABC * 100
df_dedup['price_variance_pct'] = (
    (df_dedup['Contract Amount'] - df_dedup['Approved Budget of the Contract']) / 
    df_dedup['Approved Budget of the Contract'] * 100
).replace([np.inf, -np.inf], np.nan)


print("Derived feature:")
print(df_dedup[['price_variance_pct']].describe())

Derived feature:
       price_variance_pct
count       127459.000000
mean            -8.273940
std             19.163945
min           -100.000000
25%             -4.250746
50%             -0.635595
75%             -0.088183
max             33.333333


In [23]:
# ============================================
# FIX: Drop contracts with zero budget (causing NaN in price_variance_pct)
# ============================================

zero_budget_count = (df_dedup['Approved Budget of the Contract'] == 0).sum()
if zero_budget_count > 0:
    print(f"\n⚠️ Found {zero_budget_count} contracts with Approved Budget = 0")
    print("These will be dropped as they're likely data errors")
    
    # Show examples before dropping
    print("\nExamples of contracts being dropped:")
    print(df_dedup[df_dedup['Approved Budget of the Contract'] == 0]
          [['Bid Reference No.', 'Contract Amount']].head(10))
    
    # Drop them
    df_dedup = df_dedup[df_dedup['Approved Budget of the Contract'] > 0].copy()
    print(f"Removed {zero_budget_count} zero-budget contracts")
    print(f"Remaining contracts: {len(df_dedup):,}")

# Debug: Check what caused NaNs (should now be 0)
nan_mask = df_dedup['price_variance_pct'].isna()
if nan_mask.any():
    print(f"\nDEBUG: Found {nan_mask.sum()} NaN values in price_variance_pct")
else:
    print(f"\nNo NaN values in price_variance_pct after dropping zero-budget contracts")


⚠️ Found 20 contracts with Approved Budget = 0
These will be dropped as they're likely data errors

Examples of contracts being dropped:
      Bid Reference No.  Contract Amount
4815           11314198     2.695487e+07
7496           11320526     6.655626e+08
7537           11320613     2.049966e+09
11406          11330751     8.047280e+10
11447          11330849     5.692064e+10
12627          11333728     1.237500e+06
13889          11336911     1.006679e+10
23014          11360386     4.320000e+05
26484          11369405     4.320000e+07
26638          11369801     4.814100e+07
Removed 20 zero-budget contracts
Remaining contracts: 127,459

No NaN values in price_variance_pct after dropping zero-budget contracts


## 5. Handle Missing Values

- Financial: Drop if missing
- Timeline: Fill with median
- Categorical: Keep as-is

In [24]:
## 5. Handle Missing Values

print("\n=== HANDLING MISSING VALUES ===")

# Check missing in critical columns
critical_cols = [
    'Contract Amount', 
    'Approved Budget of the Contract',
    'price_variance_pct',
]

print("Missing values before handling:")
for col in critical_cols:
    missing = df_dedup[col].isnull().sum()
    pct = missing / len(df_dedup) * 100
    print(f"{col:40} {missing:6,} ({pct:5.2f}%)")

# Drop any remaining rows with missing financials (should be 0 now)
df_clean = df_dedup.dropna(subset=['Contract Amount', 'Approved Budget of the Contract', 'price_variance_pct']).copy()

print(f"\nAfter cleaning: {len(df_clean):,} contracts")
print(f"Removed: {len(df_dedup) - len(df_clean):,} contracts with missing values")

# Verify no NaNs remain
print("\n✅ Final check - Missing values after cleaning:")
for col in critical_cols:
    missing = df_clean[col].isnull().sum()
    print(f"{col:40} {missing:6,}")


=== HANDLING MISSING VALUES ===
Missing values before handling:
Contract Amount                               0 ( 0.00%)
Approved Budget of the Contract               0 ( 0.00%)
price_variance_pct                            0 ( 0.00%)

After cleaning: 127,459 contracts
Removed: 0 contracts with missing values

✅ Final check - Missing values after cleaning:
Contract Amount                               0
Approved Budget of the Contract               0
price_variance_pct                            0


In [25]:
df_clean = df_dedup.dropna(subset=['Contract Amount', 'Approved Budget of the Contract'])

print(f"After dropping missing financials: {len(df_clean):,}")
print(f"Removed: {len(df_dedup) - len(df_clean):,}")

After dropping missing financials: 127,459
Removed: 0


## 5. Final Data Quality Check

In [26]:
# Columns for modeling
# df_clean = df_clean.reset_index()       # reset index after deduplication to get back


model_features = [
    # Financial
    'Approved Budget of the Contract',
    'Contract Amount',
    'price_variance_pct',
    
    # Categorical (will encode)
    'Procurement Mode',
    'Funding Instrument',
    'Region',
    'Awardee Size',

    'item_count',              
    'unique_unspsc_count'      
]

# Identification columns (for result interpretation)
id_cols = [
    'Bid Reference No.',
    'Procuring Entity (PE)',
    'Awardee Organization Name',
    'Notice Title',
    'UNSPSC Code',
    'UNSPSC Description'
]

# Select columns for processed dataset
output_cols = model_features + id_cols
df_processed = df_clean[output_cols].copy()

print(f"Processed dataset: {len(df_processed):,} rows × {len(df_processed.columns)} columns")

Processed dataset: 127,459 rows × 15 columns


In [27]:
# Final check
df_processed.info()

<class 'pandas.DataFrame'>
Index: 127459 entries, 0 to 127478
Data columns (total 15 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Approved Budget of the Contract  127459 non-null  float64
 1   Contract Amount                  127459 non-null  float64
 2   price_variance_pct               127459 non-null  float64
 3   Procurement Mode                 127459 non-null  str    
 4   Funding Instrument               127459 non-null  str    
 5   Region                           127459 non-null  str    
 6   Awardee Size                     123883 non-null  str    
 7   item_count                       127459 non-null  int64  
 8   unique_unspsc_count              127459 non-null  int64  
 9   Bid Reference No.                127459 non-null  object 
 10  Procuring Entity (PE)            127459 non-null  str    
 11  Awardee Organization Name        127459 non-null  str    
 12  Notice Title      

In [28]:
# Summary statistics
df_processed[model_features[:6]].describe()

,Approved Budget of the Contract,Contract Amount,price_variance_pct
count,1.274590e+05,1.274590e+05,127459.000000
mean,1.144510e+08,4.687975e+07,-8.273940
std,6.600278e+09,7.677704e+08,19.163945
min,9.000000e+00,0.000000e+00,-100.000000
25%,4.050000e+05,3.555755e+05,-4.250746
50%,1.268952e+06,1.089900e+06,-0.635595
75%,6.089792e+06,5.551424e+06,-0.088183
max,2.034791e+12,6.071617e+10,33.333333


## 6. Save Processed Data

In [29]:
# Create output directory
output_dir = Path('../data/processed')
output_dir.mkdir(exist_ok=True)

# Save with MODE in filename
output_file = output_dir / f'procurement_{MODE.lower()}.csv'
df_processed.to_csv(output_file, index=False)


print(f"✅ Saved to: {output_file}")
print(f"✅ {len(df_processed):,} rows ready for modeling")

# Show final dataset info
print("\n=== FINAL DATASET SUMMARY ===")
print(f"Mode: {MODE}")
print(f"Shape: {df_processed.shape}")
print(f"Memory: {df_processed.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Quick stats on key columns
print("\nKey statistics:")
for col in ['price_variance_pct']:
    if col in df_processed.columns:
        print(f"{col:25} mean={df_processed[col].mean():.1f} median={df_processed[col].median():.1f}")

✅ Saved to: ../data/processed/procurement_train.csv
✅ 127,459 rows ready for modeling

=== FINAL DATASET SUMMARY ===
Mode: TRAIN
Shape: (127459, 15)
Memory: 95.0 MB

Key statistics:
price_variance_pct        mean=-8.3 median=-0.6


In [30]:
## Final Verification
print("=== FINAL VERIFICATION ===")
print(f"Total contracts: {len(df_processed):,}")
print(f"Total features: {len(df_processed.columns)}")
print(f"\nSample of first contract:")
print(df_processed.iloc[0][['Bid Reference No.', 'Contract Amount', 
                             'item_count', 'unique_unspsc_count']])

=== FINAL VERIFICATION ===
Total contracts: 127,459
Total features: 15

Sample of first contract:
Bid Reference No.       7575137
Contract Amount        132840.0
item_count                    0
unique_unspsc_count           1
Name: 0, dtype: object
